#Doing Marketing Campaigns Analysis using DS Techniques

In [ ]:
#Import the Librarys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Imported the Module")


In [ ]:
#Load the Data

dataset_path=r"datasets\marketing_data.csv"

df_marketingData=pd.read_csv(dataset_path)

df_marketingData

In [ ]:
#Dt_Customer and income to verify their accurate importation
#df_marketingData.info()

#Dt_Customer should be of DateTime Type
df_marketingData["Dt_Customer"]=pd.to_datetime(df_marketingData["Dt_Customer"], format='%m/%d/%y', errors='coerce') #Conversion should be based on the data available like it is stored as 11/29/12 means format should be %m/%d/%y

#df_marketingData.info() #Dt_Customer DataType to Dt_Customer 2240 non-null  datetime64[ns]

#Now Move to income column, lets create a new column as Income_Cleaned

df_marketingData["Income_Cleaned"]=df_marketingData[' Income '].str.replace('$','').str.replace(',','').astype(float)

#df_marketingData.info() #income cleaned became float
 

In [ ]:
#Some Customers have missing Income_Cleaned Values, Conduct missing value imputation considering that customer with similar education and marital stats tend to have comparable yearly incomes on average

#df_marketingData["Income_Cleaned"].isnull().sum()  #24 rows is having empty income

df_marketingData["Income_Cleaned"]=df_marketingData.groupby(['Education',"Marital_Status"])['Income_Cleaned'].transform(lambda X:X.fillna(X.median()))

df_marketingData["Income_Cleaned"].isnull().sum()  #0 rows is having empty income

#df_marketingData.head()

In [ ]:
#Create variables to represent the total number of children, age

df_marketingData["Total_Children"]=df_marketingData["Kidhome"] + df_marketingData["Teenhome"]

from datetime import datetime

df_marketingData["Age"]=datetime.now().year-df_marketingData['Year_Birth']

df_marketingData


In [ ]:
#total Spending. 

df_marketingData.columns
# total spending.
df_marketingData['Total_Spending']=df_marketingData[['MntWines',
       'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds']].sum(axis=1)
df_marketingData['Total_Spending']

In [ ]:
#total purchases

df_marketingData.columns
df_marketingData["Total_Purchases"]=df_marketingData[['NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases']].sum(axis=1)

df_marketingData["Total_Purchases"]

Generate Box Plots and histograms to gain insights into the distributions and identify the outliers and implement outlier treatment as needed

In [ ]:
plt.figure()
plt.boxplot(df_marketingData['Income_Cleaned'])
plt.title("Boxplot of Income")
plt.ylabel("Income")
plt.show()

In [ ]:
#IQR talks outlies statistically
def detect_outlier(data,column):
  Q1=data[column].quantile(0.25)
  Q3=data[column].quantile(0.75)
  IQR=Q3-Q1
  lower_bound = Q1-1.5*IQR
  upper_bound = Q3+1.5*IQR
  outlier=data[(data[column]>upper_bound) | (data[column]<lower_bound)]
  print(f"Q1 {Q1} Q3 {Q3}")
  print(f"Lower Bound {lower_bound} Upper Bound {upper_bound}")
  print(f'Number of outliers detected in {column}:',len(outlier))
  return outlier
outlier_rows = detect_outlier(df_marketingData,'Income_Cleaned')

In [ ]:
outlier_rows=detect_outlier(df_marketingData,"Income_Cleaned")
outlier_rows

In [ ]:
#in typical finance or stock data we go with quantile taking last 1% and remove it
q=df_marketingData["Income_Cleaned"].quantile(0.99)
df_marketingData=df_marketingData[df_marketingData["Income_Cleaned"]<q]
df_marketingData.shape

In [ ]:
# Create box plot for Age
df_marketingData.boxplot(column='Age')
plt.show()

In [ ]:
q=df_marketingData["Age"].quantile(0.99)
print(q)

In [ ]:
df_marketingData=df_marketingData[df_marketingData["Age"]<q]
print(df_marketingData.shape)

df_marketingData.boxplot(column='Age')
plt.show()

5. Apply Ordinal and one-hot encoding based on the various types of Categorical variables

In [ ]:
#check for how many catrgorical variables

df_marketingData.columns #2 categorical variable, 1. Education & another one is Marital_status

df_marketingData["Education"].value_counts(), #Count against Eduction, typical group by

df_marketingData["Education"].unique()

In [ ]:
#Categorical Data of Eduction 

education_order=['Basic','2n Cycle','Graduation',  'Master', 'PhD' ]

#Map the Eduction Order

df_marketingData["Education_Encoded"]=df_marketingData["Education"].map({'Basic':0,'2n Cycle':1,'Graduation':2,  'Master':3, 'PhD':4})

df_marketingData["Education_Encoded"]

In [47]:
#Categorical Value of Marital_status

df_marketingData["Marital_Status"].value_counts(), #Count against Marital Status, typical group by

df_marketingData["Marital_Status"].unique()

array(['Divorced', 'Single', 'Married', 'Together', 'Widow', 'YOLO',
       'Alone', 'Absurd'], dtype=object)

In [ ]:
#Apply One Hot Encoding for Marital Status Column

df_marketingData=pd.get_dummies(df_marketingData,columns=["Marital_Status"])

df_marketingData